# Benchmark 1 – Replication of RF for Antimicrobial Resistance Prediction from MALDI-TOF Spectra

## Objective

In this notebook, we implement the first benchmark model of the project:  
the replication of the random forest described in:

> Astudillo, C. A., López-Cortés, X. A., Ocque, E., & Manríquez-Troncoso, J. M. (2024).  
> *Multi-label classification to predict antibiotic resistance from raw clinical MALDI-TOF mass spectrometry data*.  
> Scientific Reports, 14, 31283.  
> https://doi.org/10.1038/s41598-024-82697-w


## Background

The referenced study proposes a multi-label classification framework to predict antimicrobial resistance (AMR) from MALDI-TOF mass spectrometry data. The authors benchmarked several machine learning algorithms aming which we find RF achieving competitive performance in terms of Weighted F1-score (WF1), particularly in multi-label scenarios.


# Imports and Configuration

In [35]:
# Imports

import pickle
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    hamming_loss,
    make_scorer
)
from sklearn.ensemble import RandomForestClassifier
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

warnings.filterwarnings("ignore")

# Load pickle
pickle_path = "../data/DRIAMS_A_AMR_paper_replication.pkl"

with open(pickle_path, "rb") as f:
    payload = pickle.load(f)

# Raw arrays
X_raw = payload["data"]
amr_raw = payload["amr"]
antibiotics = payload["antibiotics"]
labels_raw = payload["label"]

print("Initial shapes:")
print("X:", X_raw.shape)
print("AMR:", amr_raw.shape)
print("Labels:", len(labels_raw))
print("Antibiotics:", len(antibiotics))

Initial shapes:
X: (14925, 1600)
AMR: (14925, 9)
Labels: 14925
Antibiotics: 9


# 1. Preprocessing

## 1.1 Build one dataframe 

For each species:

- Identify the indices of its antibiotics
- Extract only those AMR columns
- Preserve correct column order

In [36]:
# Align everything together

df_features = pd.DataFrame(X_raw)
df_amr = pd.DataFrame(amr_raw, columns=antibiotics)
df_species = pd.DataFrame(labels_raw, columns=["species"])

full_df = pd.concat([df_features, df_amr, df_species], axis=1)

print("Full dataframe shape (before cleaning):", full_df.shape)

Full dataframe shape (before cleaning): (14925, 1610)


## 1.2 Remove Duplicate Samples 

According to the paper, duplicated samples must be removed to avoid bias.

We define duplicates as:
- Identical MALDI feature vectors
- Identical AMR resistance profiles

We remove duplicates using pandas.

In [37]:
# Remove duplicates

full_df = full_df.drop_duplicates()
print("After duplicate removal:", full_df.shape)


After duplicate removal: (14924, 1610)


## 2.3 Define Species-Specific Antibiotic Subsets

Each species must be analyzed using only the antibiotics that
were tested for that organism in the paper.

We explicitly define the antibiotic subsets per species.

## 2.4 Drop NaNs

Drop samples with incomplete AMR profile / species


In [40]:
# Global antibiotic list

global_antibiotics = list(antibiotics)

print("\nGlobal antibiotic order:")
for i, ab in enumerate(global_antibiotics):
    print(i, "→", ab)


# Species-specific panels

species_antibiotics = {
    "Staphylococcus_Aureus": [
        "Oxacillin", "Clindamycin", "Fusidic acid"
    ],
    "Escherichia_Coli": [
        "Ciprofloxacin", "Ceftriaxone",
        "Piperacillin-Tazobactam", "Cefepime"
    ],
    "Klebsiella_Pneumoniae": [
        "Ciprofloxacin", "Ceftriaxone",
        "Imipenem", "Meropenem"
    ],
    "Pseudomonas_Aeruginosa": [
        "Ciprofloxacin", "Imipenem", "Meropenem"
    ]
}


Global antibiotic order:
0 → Oxacillin
1 → Clindamycin
2 → Fusidic acid
3 → Ciprofloxacin
4 → Ceftriaxone
5 → Piperacillin-Tazobactam
6 → Cefepime
7 → Imipenem
8 → Meropenem


In [41]:
results = {}

for species, ab_list in species_antibiotics.items():

    print("\n==============================")
    print("Species:", species)
    print("==============================")

    df_species_subset = full_df[full_df["species"] == species]

    print("Before NaN removal:", df_species_subset.shape)

    # Seleccionar solo columnas relevantes
    feature_cols = full_df.columns[:X_raw.shape[1]]
    species_df = df_species_subset[list(feature_cols) + ab_list]

    # Ahora sí: eliminar NaN SOLO en sus antibióticos
    species_df = species_df.dropna()

    print("After NaN removal:", species_df.shape)

    X_species = species_df.iloc[:, :X_raw.shape[1]].values
    amr_species = species_df[ab_list].values

    results[species] = {
        "X": X_species,
        "amr": amr_species,
        "antibiotics": ab_list
    }


Species: Staphylococcus_Aureus
Before NaN removal: (3791, 1610)
After NaN removal: (3556, 1603)

Species: Escherichia_Coli
Before NaN removal: (4990, 1610)
After NaN removal: (4663, 1604)

Species: Klebsiella_Pneumoniae
Before NaN removal: (2869, 1610)
After NaN removal: (2813, 1604)

Species: Pseudomonas_Aeruginosa
Before NaN removal: (3274, 1610)
After NaN removal: (2262, 1603)


## 2.5 Construct LPS Patterns and Remove Rare Combinations (<10)

For each species:

- Construct the LPS pattern string from its antibiotic panel.
- Count how many times each resistance combination appears.
- Remove combinations with fewer than 10 samples.

This step must be applied independently per species.

In [46]:
# =========================
# Remove rare LPS patterns (<10)
# =========================

filtered_results = {}

for species, data_dict in results.items():

    print("\n----------------------------------")
    print("Species:", species)
    print("----------------------------------")

    X_species = data_dict["X"]
    amr_species = data_dict["amr"]

    # Construct LPS pattern strings
    patterns = np.array([
        "".join(map(str, row.astype(int)))
        for row in amr_species
    ])

    pattern_series = pd.Series(patterns)

    print("Total samples:", len(pattern_series))
    print("Unique patterns:", pattern_series.nunique())

    counts = pattern_series.value_counts()

    print("\nTop patterns:")
    print(counts.head())

    # Keep only patterns with >=10 samples
    valid_patterns = counts[counts > 10].index

    mask_valid = pattern_series.isin(valid_patterns)

    X_filtered = X_species[mask_valid]
    amr_filtered = amr_species[mask_valid]
    patterns_filtered = patterns[mask_valid]

    print("\nAfter filtering:")
    print("Remaining samples:", X_filtered.shape[0])
    print("Remaining patterns:", len(set(patterns_filtered)))

    filtered_results[species] = {
        "X": X_filtered,
        "amr": amr_filtered,
        "patterns": patterns_filtered,
        "antibiotics": data_dict["antibiotics"]
    }


----------------------------------
Species: Staphylococcus_Aureus
----------------------------------
Total samples: 3556
Unique patterns: 8

Top patterns:
000    2430
100     423
010     286
110     190
001     111
Name: count, dtype: int64

After filtering:
Remaining samples: 3556
Remaining patterns: 8

----------------------------------
Species: Escherichia_Coli
----------------------------------
Total samples: 4663
Unique patterns: 13

Top patterns:
0000    3006
1000     530
1101     497
0101     146
1100     116
Name: count, dtype: int64

After filtering:
Remaining samples: 4649
Remaining patterns: 11

----------------------------------
Species: Klebsiella_Pneumoniae
----------------------------------
Total samples: 2813
Unique patterns: 9

Top patterns:
0000    2201
1100     281
1000     190
0100      94
1111      29
Name: count, dtype: int64

After filtering:
Remaining samples: 2795
Remaining patterns: 5

----------------------------------
Species: Pseudomonas_Aeruginosa
-------

#  Nested Cross-Validation Framework (10 Outer Folds)

We perform:

Outer loop:
    - 10-fold stratified cross-validation (80/20 splits)

Inner loop:
    - 5-fold CV on training partition
    - Bayesian optimization (200 trials)
    - Random Forest hyperparameter search

For each outer fold:
    - Best model is retrained on full training fold
    - Evaluated on held-out test fold

Metrics computed:
    - Accuracy
    - Hamming Loss
    - Weighted F1

In [47]:
def multilabel_f1_wrapper(true, pred, average="weighted"):
    if isinstance(true, list):
        true = np.array(true)
    elif isinstance(true, pd.DataFrame):
        true = true.to_numpy()

    if isinstance(pred, list):
        pred = np.array(pred)
    elif isinstance(pred, pd.DataFrame):
        pred = pred.to_numpy()

    column = 0
    total = 0

    while column < true[0].size:
        total += f1_score(true[:, column], pred[:, column], average=average)
        column += 1

    return total / column

In [48]:
lc_global = LabelEncoder()

def lps_to_multilabel_instance(lps_instance):
    multilabel_instance = []
    for result in lps_instance[0]:
        multilabel_instance.append(int(result))
    return multilabel_instance

def lps_to_multilabel_list(lps_list):
    multilabel_list = []
    for lps_instance in lps_list:
        multilabel_list.append(
            lps_to_multilabel_instance(
                lc_global.inverse_transform([lps_instance])
            )
        )
    return multilabel_list

def lps_f1_wrapper(true, pred, average="weighted"):
    non_lps_true = lps_to_multilabel_list(true)
    non_lps_pred = lps_to_multilabel_list(pred)
    return multilabel_f1_wrapper(non_lps_true, non_lps_pred, average=average)

In [49]:
N_CV = 5
N_ITER = 200

def optimize_rf_exact(train_X, train_y, scoring):

    search_space = {
        "n_estimators": Integer(1, 1000),
        "max_depth": Integer(1, 10),
        "min_samples_leaf": Integer(1, 10),
        "bootstrap": Categorical([False, True]),
        "random_state": Categorical([0])
    }

    opt = BayesSearchCV(
        RandomForestClassifier(),
        search_space,
        n_iter=N_ITER,
        cv=N_CV,
        random_state=0,
        n_jobs=10,
        n_points=2,
        verbose=1
    )

    opt.scoring = scoring
    opt.fit(train_X, train_y)

    return opt

In [50]:
outer_results = {}

for species, data_dict in filtered_results.items():

    print("\n==============================")
    print("Species:", species)
    print("==============================")

    X = data_dict["X"]
    y_multi = data_dict["amr"]        # matriz binaria
    patterns = data_dict["patterns"] # strings LPS
    antibiotics = data_dict["antibiotics"]

    outer_cv = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    results_binary = []
    results_lps = []

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, patterns)):

        print(f"\nOuter Fold {fold+1}/10")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train_multi, y_test_multi = y_multi[train_idx], y_multi[test_idx]
        y_train_patterns = patterns[train_idx]
        y_test_patterns = patterns[test_idx]

        # StandardScaler
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # 1. Independent Binary Classifiers

        binary_predictions = []

        for ab_idx in range(y_train_multi.shape[1]):

            y_train_bin = y_train_multi[:, ab_idx]

            opt = optimize_rf_exact(
                X_train,
                y_train_bin,
                scoring="f1_weighted"
            )

            best_model = opt.best_estimator_
            best_model.fit(X_train, y_train_bin)

            pred_bin = best_model.predict(X_test)
            binary_predictions.append(pred_bin)

        binary_predictions = np.array(binary_predictions).T

        # Métricas sobre problema binario multilabel
        acc_bin = accuracy_score(y_test_multi, binary_predictions)
        ham_bin = hamming_loss(y_test_multi, binary_predictions)
        f1_bin = multilabel_f1_wrapper(y_test_multi, binary_predictions)

        results_binary.append((acc_bin, ham_bin, f1_bin))

        # 2. LPS Multiclass

        le = LabelEncoder()
        y_train_lps = le.fit_transform(y_train_patterns)
        lc_global.classes_ = le.classes_

        opt = optimize_rf_exact(
            X_train,
            y_train_lps,
            scoring=make_scorer(lps_f1_wrapper)
        )

        best_model = opt.best_estimator_
        best_model.fit(X_train, y_train_lps)

        pred_lps = best_model.predict(X_test)

        # Decodificación a multilabel binario
        decoded = le.inverse_transform(pred_lps)
        pred_multi = np.array([[int(c) for c in s] for s in decoded])

        acc_lps = accuracy_score(y_test_multi, pred_multi)
        ham_lps = hamming_loss(y_test_multi, pred_multi)
        f1_lps = multilabel_f1_wrapper(y_test_multi, pred_multi)

        results_lps.append((acc_lps, ham_lps, f1_lps))

    outer_results[species] = {
        "binary_mean": np.mean(results_binary, axis=0),
        "binary_std": np.std(results_binary, axis=0),
        "lps_mean": np.mean(results_lps, axis=0),
        "lps_std": np.std(results_lps, axis=0)
    }


Species: Staphylococcus_Aureus

Outer Fold 1/10
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling

KeyboardInterrupt: 

In [ ]:
for species, results in outer_results.items():

    print("\n===================================")
    print("Species:", species)
    print("===================================")

    print("\n--- Independent Binary ---")
    print("Accuracy:", results["binary_mean"][0], "±", results["binary_std"][0])
    print("Hamming :", results["binary_mean"][1], "±", results["binary_std"][1])
    print("F1      :", results["binary_mean"][2], "±", results["binary_std"][2])

    print("\n--- LPS ---")
    print("Accuracy:", results["lps_mean"][0], "±", results["lps_std"][0])
    print("Hamming :", results["lps_mean"][1], "±", results["lps_std"][1])
    print("F1      :", results["lps_mean"][2], "±", results["lps_std"][2])